In [ ]:
from __future__ import annotations

import time
from typing import Any, Dict, Optional, Tuple

from kaggle.api.kaggle_api_extended import KaggleApi


def _as_dict(x: Any) -> Dict[str, Any]:
    """Kaggle API 有时返回 dict，有时返回 Submission 对象；这里统一成 dict。"""
    if isinstance(x, dict):
        return x
    # Submission 对象通常有 __dict__ 或属性
    d = {}
    for k in dir(x):
        if k.startswith("_"):
            continue
        try:
            v = getattr(x, k)
        except Exception:
            continue
        # 过滤掉方法
        if callable(v):
            continue
        d[k] = v
    # 如果对象本身也有 __dict__，合并一下
    if hasattr(x, "__dict__"):
        d.update(getattr(x, "__dict__", {}))
    return d


def submit_and_get_public_score(
    competition: str,
    submission_path: str,
    message: str = "auto-submit",
    poll_interval_sec: int = 15,
    timeout_sec: int = 15 * 60,
) -> Tuple[Optional[str], Optional[float], Dict[str, Any]]:
    """
    提交并获取 public score（若超时则 score 为 None）。
    返回: (submission_date, public_score, latest_submission_raw_dict)
    """
    api = KaggleApi()
    api.authenticate()

    # 1) 提交
    api.competition_submit(
        file_name=submission_path,
        message=message,
        competition=competition,
    )

    # 2) 轮询 submissions
    start = time.time()
    last_print = None

    while True:
        # 关键修复点：competition_submissions（不是 competitions_submissions）
        subs = api.competition_submissions(competition)
        if not subs:
            time.sleep(poll_interval_sec)
            continue

        latest_raw = subs[0]  # 通常是最新的一条
        latest = _as_dict(latest_raw)

        # 常见字段名：status, publicScore, date (不同版本可能大小写/命名略不同)
        status = str(latest.get("status", latest.get("Status", ""))).lower()
        public_score = latest.get("publicScore", latest.get("PublicScore", None))
        submission_date = latest.get("date", latest.get("Date", None))

        marker = (submission_date, status, public_score)
        if marker != last_print:
            print(f"[Kaggle] latest: date={submission_date} status={status} publicScore={public_score}")
            last_print = marker

        # 有些时候 publicScore 是字符串
        if public_score not in (None, "", "null"):
            try:
                return str(submission_date) if submission_date is not None else None, float(public_score), latest
            except Exception:
                return str(submission_date) if submission_date is not None else None, None, latest

        if time.time() - start > timeout_sec:
            return str(submission_date) if submission_date is not None else None, None, latest

        time.sleep(poll_interval_sec)


# if __name__ == "__main__":


# dt, score, raw = submit_and_get_public_score(COMP, FILE, MSG, poll_interval_sec=20, timeout_sec=20*60)
# print("\n===== RESULT =====")
# print("submission_date:", dt)
# print("public_score:", score)
# print("raw_keys:", sorted(raw.keys()))


In [30]:
api = KaggleApi()
api.authenticate()
# message: str = "auto-submit",
poll_interval_sec: int = 15
timeout_sec: int = 15 * 60
# 1) 提交
competition = "churn-prediction-25-26"
submission_path = "submission.csv"
message = "local python auto submit"
api.competition_submit(
    file_name=submission_path,
    message=message,
    competition=competition,
)


100%|██████████| 28.4k/28.4k [00:00<00:00, 48.5kB/s]


{"message": "Successfully submitted to Churn prediction 25/26", "ref": 49002949}

In [31]:

# 2) 轮询 submissions
start = time.time()
last_print = None
subs = api.competition_submissions(competition)
# time.sleep(poll_interval_sec)
latest_raw = subs[0]  


In [26]:
import pandas as pd


In [1]:
# demo_submit.py
from kaggle_submit import submit_and_track

if __name__ == "__main__":
    info = submit_and_track(
        submission_path="submission.csv",            # 要上传的文件
        competition="churn-prediction-25-26",       # Kaggle 比赛 slug
        message="local python auto submit_ v1",         # 提交描述
        wait_for_result=True,                       # 等待公共分数
        poll_interval=30,                           # 轮询间隔（秒）
        max_polls=8,                                # 初始轮询次数
        log_path="submission_log.csv",              # 记录日志
        extra_meta={"best_threshold": 0.53},        # 可选：附加元数据
    )
    print("Final submission info:", info)


Submitting submission.csv to churn-prediction-25-26 with note: local python auto submit_ v1


100%|██████████| 28.4k/28.4k [00:00<00:00, 60.9kB/s]


Poll 1/8: status=complete score=0.61147
Final submission info: {'ref': 49003577, 'description': 'local python auto submit_ v1', 'status': 'complete', 'score': 0.61147, 'fileName': 'submission.csv', 'submissionDate': datetime.datetime(2025, 12, 12, 17, 21, 11, 157000), 'error': '', 'teamName': 'Yuhan Wu', 'submittedBy': 'yuhanwu1003', 'isFrozen': True}


In [39]:
(
    latest_raw.public_score  is not None
    and latest_raw.public_score  != ''
)

False

In [ ]:
latest_raw.public_score is not None and is not ''


# while True:
#     # 关键修复点：competition_submissions（不是 competitions_submissions）
#     subs = api.competition_submissions(competition)
#     if not subs:
#         time.sleep(poll_interval_sec)
#         continue

#     latest_raw = subs[0]  # 通常是最新的一条
#     latest = _as_dict(latest_raw)

#     # 常见字段名：status, publicScore, date (不同版本可能大小写/命名略不同)
#     status = str(latest.get("status", latest.get("Status", ""))).lower()
#     public_score = latest.get("publicScore", latest.get("PublicScore", None))
#     submission_date = latest.get("date", latest.get("Date", None))

#     marker = (submission_date, status, public_score)
#     if marker != last_print:
#         print(f"[Kaggle] latest: date={submission_date} status={status} publicScore={public_score}")
#         last_print = marker

#     # 有些时候 publicScore 是字符串
#     if public_score not in (None, "", "null"):
#         try:
#             return str(submission_date) if submission_date is not None else None, float(public_score), latest
#         except Exception:
#             return str(submission_date) if submission_date is not None else None, None, latest

#     if time.time() - start > timeout_sec:
#         return str(submission_date) if submission_date is not None else None, None, latest

#     time.sleep(poll_interval_sec)


SyntaxError: invalid syntax (1433061293.py, line 1)